# Fine-tune LFM2.5-Audio sur le tool calling (EN) — L4

Pipeline : recuperer le dataset audio (HF) -> **analyser** -> packer -> **entrainer** (LoRA
backbone, encodeur + tetes audio geles) avec **wandb** + push HF reguliers -> **eval gold**
(audio -> tool call) sur voix held-out.

## Quelles metriques regarder ? (et comment savoir si ca apprend proprement)

**Pendant l'entrainement (wandb)** :
- `train/text_loss` et `train/text_ppl` (= exp(text_loss)) : **LE** signal direct -> à quel
  point le modele predit le TEXTE du tool call. `text_ppl` doit **descendre regulierement**
  (vers quelques unites). C'est plus parlant que la loss totale (qui melange audio).
- `train/grad_norm` : stabilite. Doit rester borne ; des pics enormes = instabilite
  (baisser le lr ; le clipping a 1.0 est deja actif).
- `train/lr` : warmup puis cosine (sanity check du scheduler).
- `val/text_ppl` : generalisation. Si `val` **remonte** alors que `train` descend ->
  **surapprentissage** -> arreter plus tot / moins de steps / plus de donnees.

**Eval gold (le verdict tool calling)** — genere depuis l'audio puis score BFCL :
- `parse_rate` : spans `<|tool_call_*|>` bien formes (vise > 0.98).
- `relevance_accuracy` : appelle quand il faut / s'abstient sinon (inclut les negatifs).
- `name_accuracy` : bon outil (vise > 0.90).
- `call_accuracy` (arg tolerant) : bon outil + bon argument (vise > 0.75).

**Verdict « il apprend proprement »** = `text_ppl` baisse + `val` ne diverge pas + les
metriques gold (name/relevance/call) **montent** d'un checkpoint a l'autre.


## 1. Repo + dependances (liquid-audio, peft, accelerate, wandb, datasets)

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/Rcarvalo/finetuning_s2s_toolcalling"   # <-- ton repo
BRANCH   = "claude/blissful-tesla-7i1yky"                              # <-- ta branche
WORK     = "/content/finetuning_s2s_toolcalling"
if not os.path.exists(WORK):
    subprocess.run(["git", "clone", REPO_URL, WORK], check=True)
subprocess.run(["git", "-C", WORK, "fetch", "origin", BRANCH], check=True)
subprocess.run(["git", "-C", WORK, "checkout", BRANCH], check=True)
subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{WORK}[train,tooldata]"], check=True)
os.chdir(WORK); sys.path.insert(0, WORK + "/src"); sys.path.insert(0, WORK + "/scripts")
print("repo:", WORK)

## 2. Connexions + parametres

In [ ]:
from huggingface_hub import login
login()                                              # token HF (write)
import wandb; wandb.login()                          # cle wandb

HF_DATASET    = "Rcarvalo/tc-en-audio-toolcalling"   # <-- dataset audio (push par build_*)
HUB_ADAPTER   = "Rcarvalo/lfm25-tc-en-adapter"       # <-- ou pousser l'adaptateur LoRA (prive)
WANDB_PROJECT = "lfm25-toolcalling-en"
RUN_NAME      = "phase_en_v1"
print("dataset:", HF_DATASET, "| adapter:", HUB_ADAPTER)

## 3. Recuperer le dataset (HF -> JSONL + WAV)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "scripts/hf_to_dialogues.py",
                "--repo-id", HF_DATASET, "--out", "data/tc_en"], check=True)

## 4. Analyser le dataset (« ai-je un bon dataset ? »)

In [ ]:
import analyze_dataset as ad
rows = ad.load_rows("data/tc_en/train.jsonl")
rep = ad.analyze(rows, "data/tc_en/audio_train")
import json; print(json.dumps(rep["distribution"], indent=2)); print(json.dumps(rep.get("audio", {}), indent=2))
issues = ad.flags(rep)
print("\n" + ("✅ dataset sain — aucun drapeau" if not issues else "⚠️ drapeaux:\n - " + "\n - ".join(issues)))

## 5. Split train/val + packing (LFM2AudioChatMapper)

In [ ]:
import json, random, subprocess, sys
rows = [l for l in open("data/tc_en/train.jsonl")]
random.Random(0).shuffle(rows)
k = max(1, int(len(rows) * 0.05))
open("data/tc_en/val.jsonl", "w").writelines(rows[:k])
open("data/tc_en/train_only.jsonl", "w").writelines(rows[k:])
print(f"train {len(rows)-k} / val {k}")

import s2s_toolcalling.tools.schemas as s
open("tools_en.json", "w").write(json.dumps(s.TOOLCALLING_EN_TOOL_DEFINITIONS))
for split, out in [("train_only", "datasets/tc_en_train"), ("val", "datasets/tc_en_val")]:
    subprocess.run([sys.executable, "-m", "s2s_toolcalling.data.preprocess_sft",
        "--dialogues", f"data/tc_en/{split}.jsonl", "--audio-root", "data/tc_en/audio_train",
        "--output", out, "--tool-definitions", "tools_en.json",
        "--assistant-audio-mode", "sequential"], check=True)

## 6. Entrainer (LoRA backbone ; wandb + push HF reguliers)

In [ ]:
from s2s_toolcalling.training.train_sft import TrainConfig, build_trainer
cfg = TrainConfig.from_yaml("configs/phase_en_toolcalling.yaml")
cfg.train_dataset, cfg.val_dataset = "datasets/tc_en_train", "datasets/tc_en_val"
cfg.wandb_project, cfg.wandb_run_name, cfg.hub_repo = WANDB_PROJECT, RUN_NAME, HUB_ADAPTER

n = sum(1 for _ in open("data/tc_en/train_only.jsonl"))      # ~2 epoques
cfg.max_steps = max(300, min(cfg.max_steps, 2 * n // cfg.batch_size))
print("max_steps:", cfg.max_steps, "| push tous les", cfg.push_interval, "steps")

trainer = build_trainer(cfg)
trainer.train()       # suis text_ppl / grad_norm / val_text_ppl sur wandb

## 7. Eval GOLD : audio -> tool call sur voix held-out (BFCL)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "scripts/eval_audio_toolcalling.py",
    "--backend", "liquid", "--checkpoint", "LiquidAI/LFM2.5-Audio-1.5B",
    "--adapter", "outputs/phase_en_toolcalling/adapter",
    "--cases", "data/tc_en/test.jsonl", "--audio-root", "data/tc_en/audio_test",
    "--out", "eval_tc_en.jsonl", "--arg-match", "token_f1"], check=True)

## Lire le resultat
Le bloc `summary` donne `parse_rate / relevance_accuracy / name_accuracy / call_accuracy`.
Seuils v1 : **name > 0.90, relevance > 0.85, call(arg tolerant) > 0.75, parse > 0.98**.

- **name/relevance hauts mais call plus bas** : le routage marche, c'est l'**argument** (query/
  question) qui derape -> plus de diversite de formulations dans les donnees, ou garder
  `--arg-match token_f1`.
- **relevance bas** : sur/sous-appelle -> plus de negatifs / cas limites.
- **parse bas** : spans mal formes -> rare (le format est natif) ; verifier le system prompt.

Compare aussi a la baseline (sans `--adapter`) pour mesurer le gain du fine-tune. L'adaptateur
est sur `HUB_ADAPTER` ; merge/export pour vLLM : voir README (export_checkpoint).